# 03 · Bonus A: reinforcement learning with verifiable rewards (GRPO)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/57Ajay/modeltune/blob/main/notebooks/03_bonus_grpo_gsm8k.ipynb)

Companion to [chapter 12](https://github.com/57Ajay/modeltune/blob/main/course/12_rl_grpo_reasoning.md). No demonstrations, no preference pairs: the model
samples 8 answers per math question, a Python function checks them, and the model is pushed toward the answers that
beat their group's average. About 40–60 min on a T4 (estimate).

## 0. Setup
Runtime → Change runtime type → **T4 GPU**. Any newer GPU works too; the code picks fp16 or bf16 by itself.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Get the course code, then install the pinned libraries from pyproject.toml with uv.
# Colab already ships a CUDA build of PyTorch; `uv pip install --system` keeps it instead of downloading another.
!test -d modeltune || git clone -q https://github.com/57Ajay/modeltune.git
%cd modeltune
!git pull -q
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q -r pyproject.toml

In [ ]:
# Hugging Face token: add HF_TOKEN in Colab's key icon (Secrets) panel, with WRITE access to push models.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import login
    login()
# Pushing to someone else's namespace fails: change hub.repo_id in the configs (or override it) to YOUR username.

## 1. Baseline: accuracy and format-following before RL

In [ ]:
!python -m modeltune.evaluate configs/bonus_grpo_qwen3_gsm8k.yaml --base

## 2. Train with GRPO
In the logs: `reward` should trend up; `reward_std` must stay > 0 (if every answer in a group gets the same reward, there is nothing to learn from); watch `completions/mean_length`.

In [ ]:
!python -m modeltune.train configs/bonus_grpo_qwen3_gsm8k.yaml

## 3. After

In [ ]:
!python -m modeltune.evaluate configs/bonus_grpo_qwen3_gsm8k.yaml

## Experiments
1. Remove `format_reward` from `rewards:`. Does accuracy still improve? Does the format survive?
2. `train.num_generations=4` vs `16` (adjust `per_device_train_batch_size` to match). Group size vs signal quality.
3. `train.temperature=0.6`: less diversity means fewer groups with mixed rewards, which means a weaker signal.
4. Write a new reward in `modeltune/rewards.py` (e.g. penalize answers > 200 words) and add it to the config.